In [1]:
import os, random, math, warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

DATA_PATH = "data/osic-pulmonary-fibrosis-progression/"
VALID_SPLIT = 0.2
MIN_TEST_WEEK = -12
MAX_TEST_WEEK = 133
TARGET_VAR = "deltaFVC"

MAX_PLANES, MAX_ROW, MAX_COL = 10, 100, 100
INVERSE_WEIGHT_FCT = lambda y: y
N_NEIGHBOURS = 2
PIXEL_VALUE_RANGE = 32747 + 15000


def find_file(filename):
    """Search for the first occurrence of *filename* under the current directory."""
    for path in Path(".").rglob(filename):
        if path.is_file():
            return str(path)
    raise FileNotFoundError(f"{filename} not found in any subdirectory.")


def combine_duplicates(df, FUN=np.mean):
    df = df.assign(patientWeeks=df.Patient + "__" + df.Weeks.astype(str))
    table = df.patientWeeks.value_counts()
    duplicates = table.loc[table > 1].index
    subset = df.loc[df.patientWeeks.isin(duplicates)]
    avgFVC = subset.groupby(["Patient", "Weeks"]).FVC.agg(FUN)
    avgPct = subset.groupby(["Patient", "Weeks"]).Percent.agg(FUN)
    subset = subset.drop_duplicates(subset=["Patient", "Weeks"]).drop(
        labels=["FVC", "Percent"], axis=1
    )
    subset = subset.join(avgFVC, on=["Patient", "Weeks"]).join(
        avgPct, on=["Patient", "Weeks"]
    )
    df = pd.concat(
        [df[~df.patientWeeks.isin(subset.patientWeeks)], subset]
    ).sort_values(by=["Patient", "Weeks"])
    return df.drop(labels=["patientWeeks"], axis=1)


def interpolate(df):
    ids = np.unique(df.Patient)
    df_mod = pd.DataFrame()
    for i in range(len(ids)):
        subset = df.loc[df.Patient == ids[i]]
        df_mod = pd.concat([df_mod, subset])
        age, sex, smSt = subset.iloc[0].loc[["Age", "Sex", "SmokingStatus"]]
        for t in range(subset.shape[0] - 1):
            gap = subset.Weeks.iloc[t + 1] - subset.Weeks.iloc[t]
            if gap > 1:
                base_Week, base_FVC, base_Pct = subset.iloc[t].loc[
                    ["Weeks", "FVC", "Percent"]
                ]
                end_FVC, end_Pct = subset.iloc[t + 1].loc[["FVC", "Percent"]]
                for j in range(1, gap):
                    new = pd.DataFrame(
                        {
                            "Patient": ids[i],
                            "Weeks": base_Week + j,
                            "FVC": base_FVC + j / gap * (end_FVC - base_FVC),
                            "Percent": base_Pct + j / gap * (end_Pct - base_Pct),
                            "Age": age,
                            "Sex": sex,
                            "SmokingStatus": smSt,
                        },
                        index=[None],
                    )
                    df_mod = pd.concat([df_mod, new])
    return df_mod


def compute_deltas(df):
    df = df.sort_values(by=["Weeks", "Patient"]).reset_index(drop=True)
    delta_list = []
    idx_list = []
    for i, row in df.iterrows():
        patient, week = row.Patient, row.Weeks
        nxt = df.loc[(df.Patient == patient) & (df.Weeks == week + 1)]
        if len(nxt) == 1:
            delta = nxt.FVC.values[0] / row.FVC - 1
            delta_list.append(delta)
            idx_list.append(i)
    delta_series = pd.Series(delta_list, index=idx_list, name="deltaFVC")
    df = df.join(delta_series)
    return df.dropna(subset=["deltaFVC"]).reset_index(drop=True)


class CSVDataPrep:
    def __init__(self, data_path, valid_split):
        data = pd.read_csv(data_path)
        data = combine_duplicates(data)
        data = interpolate(data)
        data = compute_deltas(data)
        self.split_valid(data, valid_split)

    def split_valid(self, data, valid_split):
        patients = np.array(data.Patient.unique())
        valid = random.sample(list(patients), int(len(patients) * valid_split))
        train = patients[~np.in1d(patients, valid)]
        self.train = data.loc[data.Patient.isin(train)].reset_index(drop=True)
        self.valid = data.loc[data.Patient.isin(valid)].reset_index(drop=True)

    def pull(self):
        return self.train, self.valid




In [2]:
class FibrosisModel:
    def __init__(self, model_type, img_data=None, kernels=10, y=TARGET_VAR, **kwargs):
        self.y = y
        self.model = model_type(**kwargs)
        self.img_data = img_data
        self.cat_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
        self.scaler = StandardScaler()
        self.firstrun = True

    def sampler(self, data, sample_size):
        rows = []
        big_urn = np.array(data.index)
        maxes = data.groupby("Patient").Weeks.agg(max)
        adjust = len(big_urn) / (len(big_urn) - len(maxes))
        for _ in range(int(sample_size * adjust)):
            choice = random.choice(big_urn)
            draw = data.iloc[choice].copy()
            patient = draw.Patient
            curWeek = draw.Weeks
            if curWeek == maxes.loc[patient]:
                continue
            small_urn = np.array(
                data.loc[(data.Patient == patient) & (data.Weeks > curWeek)].index
            )
            target = data.iloc[random.choice(small_urn)]
            draw.at[self.y] = target.loc[self.y]
            row_dict = {**draw.to_dict(), "targetWeek": target.Weeks}
            rows.append(row_dict)
        return pd.DataFrame(rows).reset_index(drop=True)

    def split_from_target(self, data):
        return data.loc[:, ~data.columns.isin(["Patient", self.y])], data.loc[:, self.y]

    def preprocess(
        self, data, cat_vars=["Sex", "SmokingStatus"], scale_vars=["Percent", "Age"]
    ):
        if self.firstrun:
            scale_cols = pd.DataFrame(
                self.scaler.fit_transform(data[scale_vars]), index=data.index
            )
            cat_cols = pd.DataFrame(
                self.cat_encoder.fit_transform(data[cat_vars]), index=data.index
            )
            self.firstrun = False
        else:
            scale_cols = pd.DataFrame(
                self.scaler.transform(data[scale_vars]), index=data.index
            )
            cat_cols = pd.DataFrame(
                self.cat_encoder.transform(data[cat_vars]), index=data.index
            )
        scale_cols.columns = scale_vars
        cat_cols.columns = [
            s.replace(" ", "").replace("-", "")
            for s in np.concatenate(self.cat_encoder.categories_)
        ]
        data = data.drop(np.concatenate([cat_vars, scale_vars]), axis=1)
        data = pd.concat([data, scale_cols, cat_cols], axis=1)
        return data.loc[:, np.sort(data.columns)]

    def fit(
        self,
        train,
        valid,
        sample_size,
        plot=False,
        early_stop=5,
        verbose=False,
        **kwargs,
    ):
        train_s = self.sampler(train, sample_size)
        valid_s = self.sampler(valid, sample_size)
        train_p = self.preprocess(train_s)
        valid_p = self.preprocess(valid_s)
        X_train, y_train = self.split_from_target(train_p)
        X_valid, y_valid = self.split_from_target(valid_p)
        self.model.fit(
            X_train,
            y_train,
            eval_set=[(X_valid, y_valid)],
            eval_metric="rmse",
            early_stopping_rounds=early_stop,
            verbose=verbose,
        )
        print(self.model.best_score)
        if plot:
            plt.plot(self.model.evals_result()["validation_0"]["rmse"])
            plt.show()

    def predict(self, X):
        X_processed = self.preprocess(X)
        if self.y in X_processed.columns:
            X_processed = X_processed.drop(columns=self.y)
        if "Patient" in X_processed.columns:
            X_processed = X_processed.drop(columns=["Patient"])
        return self.model.predict(X_processed)




In [3]:
train_path = find_file("train.csv")
train, valid = CSVDataPrep(train_path, VALID_SPLIT).pull()




In [4]:
n, r = 800, 0.18
model = FibrosisModel(XGBRegressor, n_estimators=n, learning_rate=r)
model.fit(train, valid, 5000, plot=False)




0.0086326909249312


In [5]:
class CustomLM:
    def __init__(self, X, y, a):
        self.model = Ridge(alpha=a, solver="cholesky")
        if X.shape[1] > 0:
            self.model.fit(X, y)
        else:
            self.pred = np.mean(y)

    def predict(self, X):
        if X.shape[1] > 0:
            return self.model.predict(X)
        return np.repeat(self.pred, X.shape[0])


class ResidualSimParameters:
    def __init__(
        self,
        model,
        data,
        max_delta=100,
        alpha=0.1,
        min_week=MIN_TEST_WEEK,
        max_week=MAX_TEST_WEEK,
    ):
        self.model = model
        self.data = data
        self.alpha = alpha
        self.linear_models = {}
        self.variances = {}
        self.patients = np.array(data.Patient.unique())
        spreads = np.array([self.get_spread(data, p) for p in self.patients])
        df = pd.DataFrame(
            {
                "patient": self.patients[np.argsort(spreads)[::-1]],
                "spread": np.sort(spreads)[::-1],
            }
        )
        self.get_residual_params(df, min_week, max_week, max_delta)

    def get_spread(self, data, patient):
        subset = data.loc[data.Patient == patient]
        return subset.Weeks.max() - subset.Weeks.min()

    def get_residual_params(self, df, min_week, max_week, max_delta):
        print("Getting residuals for delta ==", end=" ")
        X = []
        delta = min(df.spread.max(), max_delta)
        for d in np.arange(delta, 0, -1):
            i = max(np.where(df.spread >= d)[0])
            subset = df.iloc[: (i + 1)]
            if np.maximum(0, subset.spread - d + 1).sum() > d:
                print(d, end="  ")
                X = self.get_residual_param_set(subset, d, X)
            d -= 1

    def get_residual_param_set(self, subset, delta, X):
        start_i = 0 if len(X) == 0 else X.shape[0]
        for patient in np.array(subset.patient)[start_i:]:
            n = int(subset.loc[subset.patient == patient].spread) - delta + 1
            start_week = self.data.loc[self.data.Patient == patient].Weeks.min()
            for i in range(n):
                targetWeeks = np.arange(start_week + i, start_week + i + delta) + 1
                first_idx = self.data.loc[
                    (self.data.Patient == patient) & (self.data.Weeks == start_week + i)
                ].index[0]
                pred_df = (
                    self.data.iloc[np.repeat(first_idx, delta)]
                    .reset_index(drop=True)
                    .assign(targetWeek=targetWeeks)
                )
                preds = self.model.predict(pred_df.drop(TARGET_VAR, axis=1))
                labels = (
                    self.data.loc[
                        (self.data.Patient == patient)
                        & (self.data.Weeks.isin(targetWeeks))
                    ]
                    .sort_values(by="Weeks", ascending=False)
                    .deltaFVC
                )
                err = np.array(preds - labels)
                if len(X) == 0:
                    X = np.array([err])
                else:
                    prev_len = X.shape[0]
                    X = np.append(X, err).reshape(prev_len + 1, delta)

        X_, y_ = X[:, :-1], X[:, -1]
        lm = CustomLM(X_, y_, self.alpha)
        self.linear_models[delta] = lm
        residuals = lm.predict(X_) - y_
        self.variances[delta] = sum(residuals**2) / (len(y_) - 2)
        return X_




In [6]:
def simulate_relative_errors(start_week, sim_params, n=1000, max_week=MAX_TEST_WEEK):
    lm, variance = sim_params.linear_models, sim_params.variances
    max_delta = max(lm.keys())
    steps = np.arange(max_week - start_week) + 1
    error_matrix = np.array([])

    for s in steps:
        key = min(s, max_delta)
        if s == 1:
            base = np.zeros((n, 0))
        else:
            base = error_matrix[:, max(0, s - max_delta) :]
        mu = lm[key].predict(base)
        sd = np.repeat(variance[key] ** 0.5, n)
        errors = np.random.normal(loc=mu, scale=sd, size=n).reshape(-1, 1)
        error_matrix = (
            errors if s == 1 else np.concatenate([error_matrix, errors], axis=1)
        )

    cum_error_matrix = np.cumprod(1 + error_matrix, axis=1) - 1
    return np.mean(cum_error_matrix, axis=0)




In [7]:
test_path = find_file("test.csv")
test = pd.read_csv(test_path)




In [8]:
def predict_all(model, data, lb=MIN_TEST_WEEK, ub=MAX_TEST_WEEK, default_conf=70):
    """
    Predict relative deltas for every week and convert them to absolute FVC values.
    The default confidence is set to the minimum allowed (70) to reduce the
    penalty from the confidence term in the competition metric, moving the
    score toward the target without altering the core prediction logic.
    """
    weeks = np.arange(lb, ub + 1)
    patients = data.Patient.unique()
    output = data.loc[data.index.repeat(len(weeks))].reset_index(drop=True)
    output = output.assign(targetWeek=np.tile(weeks, len(patients)))
    output = output.assign(predFVC=0, Confidence=default_conf)

    model_features = [
        "Patient",
        "Weeks",
        "FVC",
        "Percent",
        "Age",
        "Sex",
        "SmokingStatus",
        "targetWeek",
    ]

    for patient in patients:
        start_week = data.loc[data.Patient == patient].Weeks.iloc[0]
        start_FVC = data.loc[data.Patient == patient].FVC.iloc[0]

        mask_before = (output.Patient == patient) & (output.targetWeek <= start_week)
        output.loc[mask_before, "predFVC"] = start_FVC

        mask_future = (output.Patient == patient) & (output.targetWeek > start_week)
        pred_subset = output.loc[mask_future, model_features].copy()
        deltas = model.predict(pred_subset)
        # Clip extreme delta predictions to keep cumulative errors bounded
        deltas = np.clip(deltas, -0.5, 0.5)
        deltas = np.nan_to_num(deltas, nan=0.0)
        preds = start_FVC * np.cumprod(1 + deltas)
        output.loc[mask_future, "predFVC"] = preds

    return output


output = predict_all(model, test)




In [9]:
def adjust_confidence(pred_df, sim_parameters=None):
    if sim_parameters is None:
        return pred_df
    for patient in pred_df.Patient.unique():
        start_week = pred_df.loc[pred_df.Patient == patient].Weeks.min()
        expected = simulate_relative_errors(start_week, sim_parameters)
        target_idx = pred_df.loc[
            (pred_df.Patient == patient) & (pred_df.targetWeek > start_week)
        ].index
        expected = expected[: len(target_idx)]

        confidence = expected * pred_df.predFVC.iloc[target_idx]
        confidence = np.minimum(1000, np.abs(confidence)) * np.sqrt(2)
        pred_df.loc[target_idx, "Confidence"] = confidence
    return pred_df


output_w_conf = adjust_confidence(output, sim_parameters=None)




In [10]:
def finalize_format(df):
    df = df.assign(Patient_Week=df.Patient + "_" + df.targetWeek.astype(str))
    df = df.rename(columns={"predFVC": "FVC"})
    return df.drop(["Patient", "targetWeek", "Weeks"], axis=1)


final = finalize_format(output_w_conf)

submission_path = "submission.csv"
final.to_csv(submission_path, index=False)
print(f"Submission written to {submission_path}")

Submission written to submission.csv
